In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report
import joblib

# -----------------------------
# Step 1: Load your dataset
# -----------------------------
# Replace with your actual dataset file
df = pd.read_excel("dataset.xlsx")

# -----------------------------
# Step 2: Prepare features and labels
# -----------------------------
# Adjust column names to match your dataset
X = df[["Temp", "SPO2", "Pressure"]]   # input features

# These should already exist in your dataset as status labels
y = df[["Temp_Status", "SPO2_Status", "Compression_Status"]]

# -----------------------------
# Step 3: Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Step 4: Train multi-output model
# -----------------------------
base_model = RandomForestClassifier(random_state=42)
multi_model = MultiOutputClassifier(base_model)
multi_model.fit(X_train, y_train)

# -----------------------------
# Step 5: Evaluate
# -----------------------------
y_pred = multi_model.predict(X_test)

for i, col in enumerate(y.columns):
    print(f"\nReport for {col}:")
    print(classification_report(y_test[col], y_pred[:, i]))

# -----------------------------
# Step 6: Save model for deployment
# -----------------------------
joblib.dump(multi_model, "multi_output_model.pkl")
print("Model saved as multi_output_model.pkl")


Report for Temp_Status:
                      precision    recall  f1-score   support

Inflammation on vein       1.00      1.00      1.00        43
 Low temp therapy on       1.00      1.00      1.00        36
              Normal       1.00      1.00      1.00        41

            accuracy                           1.00       120
           macro avg       1.00      1.00      1.00       120
        weighted avg       1.00      1.00      1.00       120


Report for SPO2_Status:
              precision    recall  f1-score   support

  Low oxygen       1.00      1.00      1.00        62
      Normal       1.00      1.00      1.00        58

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120


Report for Compression_Status:
                   precision    recall  f1-score   support

Compression Issue       1.00      1.00      1.00        62
           Normal       1.00      1

In [14]:
import pandas as pd
import joblib

# Load the trained multi-output model
multi_model = joblib.load("multi_output_model.pkl")

# Ask user for input values
temp = float(input("Enter Temperature value: "))
spo2 = float(input("Enter SpO2 value: "))
force = float(input("Enter Force/Pressure value: "))

# Prepare input for prediction
X_new = pd.DataFrame([[temp, spo2, force]], columns=["Temp", "SPO2", "Pressure"])

# Predict
prediction = multi_model.predict(X_new)

# Extract results
Temp_Status, SPO2_Status, Compression_Status = prediction[0]

# Show output
print("\nPrediction Results:")
print(f"Temperature Status   : {Temp_Status}")
print(f"SPO2 Status          : {SPO2_Status}")
print(f"Compression Status   : {Compression_Status}")

Enter Temperature value:  23
Enter SpO2 value:  12
Enter Force/Pressure value:  12



Prediction Results:
Temperature Status   : Low temp therapy on
SPO2 Status          : Low oxygen
Compression Status   : Compression Issue
